# SE446 Milestone 2 — Chicago Crime Analytics with Spark + MLlib

**Group**: CrimeDataEngineers  
**Assembled by**: Abdulaziz AlSuwailim (ID: 230253)

## Group Members

| Member | ID | M2 Tasks |
|---|---|---|
| Abdulaziz AlSuwailim | 230253 | Notebook assembly, GitHub coordination |
| Sulaiman AlEiteibi | 220391 | Phase A Tasks 1-4 (`m2_phase_a_sulaiman.py`) |
| Abdulaziz AlSharif | 230055 | Tasks 1-2 (notebook), Task 11 (spark-submit) |
| Wadee Feras Kharbat | 230685 | Phase B Tasks 5-7 (`m2_spark_ml.py`) |
| Abdulaziz AlSenani | 230524 | Task 3 (notebook), Task 9 (local), Task 10 (cluster client) |

## Overview

This notebook reproduces the M1 MapReduce analyses using Spark DataFrames (Phase A, Tasks 1-4), then builds a full MLlib arrest-prediction pipeline with three classifiers (Phase B, Tasks 5-7). Phase C documents the three execution modes (Tasks 9-11).

**Auto-detection**: The setup cell detects whether `HADOOP_CONF_DIR` is set. If so, it uses `yarn` and HDFS paths; otherwise it uses `local[*]` with the local sample CSV.

In [1]:
import sys
import os
from pyspark.sql import SparkSession

# Auto-detect: cluster if HADOOP_CONF_DIR is set, else local
is_cluster = bool(os.environ.get('HADOOP_CONF_DIR', ''))

if is_cluster:
    master = 'yarn'
    phase_a_input = 'hdfs:///data/chicago_crimes.csv'
    phase_b_input = 'hdfs:///data/chicago_crimes_sample.csv'
else:
    master = 'local[*]'
    phase_a_input = 'data/chicago_crimes_sample.csv'
    phase_b_input = 'data/chicago_crimes_sample.csv'

spark = SparkSession.builder \
    .master(master) \
    .appName('M2_Spark_ML_GroupX') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

print(f'Spark Version: {spark.version}')
print(f'Master: {spark.sparkContext.master}')
print(f'Phase A input: {phase_a_input}')
print(f'Phase B input: {phase_b_input}')

Spark Version: 4.1.1
Master: local[*]
Phase A input: data/chicago_crimes_sample.csv
Phase B input: data/chicago_crimes_sample.csv


---
## Phase A: Spark DataFrame Analytics
### Reproducing M1 MapReduce analyses using Spark DataFrames and Spark SQL

**Cluster run**: `hdfs:///data/chicago_crimes.csv` (793,073 rows)  
**Local run**: `data/chicago_crimes_sample.csv` (10,000 rows)

In [2]:
from pyspark.sql.functions import col, desc, count, avg, round as spark_round

df = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .csv(phase_a_input)

df = df.select(
    col('ID'),
    col('Date'),
    col('Primary Type'),
    col('Location Description'),
    col('Arrest').cast('boolean').alias('Arrest'),
    col('Domestic').cast('boolean').alias('Domestic'),
    col('District').cast('int').alias('District'),
    col('Year').cast('int').alias('Year')
)

print('Total Rows:', df.count())

Total Rows: 10000


### Task 1: Crime Type Distribution
**Author: Abdulaziz AlSharif (ID: 230055)**

Count crimes by `Primary Type`, ordered descending. Equivalent to M1 Task 2 (MapReduce).

In [3]:
# ============================================
# Task 1: Crime Type Distribution
# Author: Abdulaziz AlSharif (ID: 230055)
# ============================================

print('Task 1: Top 10 Crime Types')
task1 = df.groupBy('Primary Type') \
    .count() \
    .orderBy(desc('count')) \
    .limit(10)

task1.show(truncate=False)

Task 1: Top 10 Crime Types
+-------------------+-----+
|Primary Type       |count|
+-------------------+-----+
|THEFT              |2054 |
|BATTERY            |1728 |
|CRIMINAL DAMAGE    |1062 |
|MOTOR VEHICLE THEFT|948  |
|ASSAULT            |878  |
|DECEPTIVE PRACTICE |799  |
|OTHER OFFENSE      |586  |
|ROBBERY            |508  |
|BURGLARY           |316  |
|WEAPONS VIOLATION  |284  |
+-------------------+-----+



#### Task 1 Analysis — M1 vs M2 Comparison

The local output above uses the 10,000-row sample. The **cluster run** (full 793,073 rows) shows:

| Rank | Crime Type | M2 Spark (cluster, 793k) | M1 MapReduce |
|---:|---|---:|---|
| 1 | THEFT | 162,688 | M1 ran on ~10k sample |
| 2 | BATTERY | 151,930 | |
| 3 | CRIMINAL DAMAGE | 91,241 | |
| 4 | NARCOTICS | 74,127 | |
| 5 | ASSAULT | 54,070 | |

M1 used a sample, so raw counts differ. The Spark DataFrame approach requires a single `groupBy().count().orderBy()` chain vs. separate mapper and reducer Python scripts in M1.

### Task 2: Location Hotspots (Spark SQL)
**Author: Abdulaziz AlSharif (ID: 230055)**

Demonstrates `spark.sql()` — same question as M1 Task 3, but using SQL-on-Spark.

In [4]:
# ============================================
# Task 2: Location Hotspots (Spark SQL)
# Author: Abdulaziz AlSharif (ID: 230055)
# ============================================

df.createOrReplaceTempView('crimes')

print('Task 2: Top 10 Location Hotspots using Spark SQL')
task2 = spark.sql("""
    SELECT `Location Description`, COUNT(*) AS total
    FROM crimes
    WHERE `Location Description` IS NOT NULL
    GROUP BY `Location Description`
    ORDER BY total DESC
    LIMIT 10
""")

task2.show(truncate=False)

Task 2: Top 10 Location Hotspots using Spark SQL
+--------------------------------------+-----+
|Location Description                  |total|
+--------------------------------------+-----+
|STREET                                |2737 |
|APARTMENT                             |1909 |
|RESIDENCE                             |1358 |
|SIDEWALK                              |536  |
|PARKING LOT / GARAGE (NON RESIDENTIAL)|362  |
|SMALL RETAIL STORE                    |234  |
|ALLEY                                 |219  |
|RESTAURANT                            |187  |
|OTHER (SPECIFY)                       |156  |
|VEHICLE NON-COMMERCIAL                |139  |
+--------------------------------------+-----+



#### Task 2 Analysis — M1 vs M2 Comparison

Cluster run (793,073 rows): STREET (248,326) — identical to M1 Task 3 which also ran on the full dataset. Both M1 MapReduce and Spark SQL produce the same counts; STREET is the #1 crime hotspot in both. Spark SQL is faster because it avoids the full disk-based shuffle that MapReduce requires.

### Task 3: Crime Trend Over Years
**Author: Abdulaziz AlSenani (ID: 230524)**

Count crimes per year. On local mode a matplotlib chart is saved; on cluster, the printed table is used as evidence.

In [5]:
# ============================================
# Task 3: Crime Trend Over Years
# Author: Abdulaziz AlSenani (ID: 230524)
# ============================================

print('Task 3: Crime Trend Over Years')
task3 = df.where(col('Year').isNotNull()) \
    .groupBy('Year') \
    .count() \
    .orderBy('Year')

task3.show(50, truncate=False)

if spark.sparkContext.master.startswith('local'):
    import matplotlib.pyplot as plt
    os.makedirs('output/m2_phase_a', exist_ok=True)
    yearly_rows = task3.collect()
    years  = [row['Year']  for row in yearly_rows]
    counts = [row['count'] for row in yearly_rows]
    plt.figure(figsize=(10, 5))
    plt.plot(years, counts, marker='o')
    plt.xlabel('Year')
    plt.ylabel('Crime Count')
    plt.title('Crime Trend Over Years (Chicago)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('output/m2_phase_a/task3_yearly_trend.png')
    plt.show()
    print('Saved chart: output/m2_phase_a/task3_yearly_trend.png')
else:
    print('Cluster mode: printed yearly table is used for Task 3 evidence.')

Task 3: Crime Trend Over Years
+----+-----+
|Year|count|
+----+-----+
|2001|4    |
|2002|2    |
|2003|1    |
|2004|6    |
|2005|19   |
|2006|4    |
|2007|7    |
|2008|16   |
|2009|5    |
|2010|5    |
|2011|7    |
|2012|9    |
|2013|10   |
|2014|16   |
|2015|28   |
|2016|20   |
|2017|49   |
|2018|28   |
|2019|36   |
|2020|25   |
|2021|83   |
|2022|135  |
|2023|9446 |
|2024|39   |
+----+-----+

Saved chart: output/m2_phase_a/task3_yearly_trend.png


#### Task 3 Analysis — M1 vs M2 Comparison

Cluster run (full 793,073 rows): 2001 had 467,301 crimes — the peak year. Both M1 MapReduce (Task 4) and M2 Spark produce **identical** yearly counts (2001: 467,301; 2002: 205,266; etc.). The large 2023 spike in the sample data (9,446 out of 10,000) indicates the sample is weighted toward recent records. The full-dataset cluster output matches M1 exactly.

### Task 4: Arrest Rate Analysis
**Author: Sulaiman AlEiteibi (ID: 220391)**

Overall arrest rate plus a breakdown by crime type — extends M1 Task 5 which only computed the total.

In [6]:
# ============================================
# Task 4: Arrest Rate Analysis
# Author: Sulaiman AlEiteibi (ID: 220391)
# ============================================

print('Task 4: Overall Arrest Rate')
overall_arrest = df.select(
    spark_round(avg(col('Arrest').cast('int')) * 100, 2).alias('overall_arrest_rate_percent')
)
overall_arrest.show()

print('Task 4: Arrest Rate by Top 10 Crime Types')
task4 = df.groupBy('Primary Type') \
    .agg(
        count('*').alias('total_crimes'),
        spark_round(avg(col('Arrest').cast('int')) * 100, 2).alias('arrest_rate_percent')
    ) \
    .orderBy(desc('total_crimes')) \
    .limit(10)

task4.show(truncate=False)

print('Highest Arrest Rates among Top Crime Types')
task4.orderBy(desc('arrest_rate_percent')).show(3, truncate=False)

print('Lowest Arrest Rates among Top Crime Types')
task4.orderBy('arrest_rate_percent').show(3, truncate=False)

Task 4: Overall Arrest Rate
+---------------------------+
|overall_arrest_rate_percent|
+---------------------------+
|                      12.83|
+---------------------------+

Task 4: Arrest Rate by Top 10 Crime Types
+-------------------+------------+-------------------+
|Primary Type       |total_crimes|arrest_rate_percent|
+-------------------+------------+-------------------+
|THEFT              |2054        |4.97               |
|BATTERY            |1728        |19.39              |
|CRIMINAL DAMAGE    |1062        |4.33               |
|MOTOR VEHICLE THEFT|948         |4.54               |
|ASSAULT            |878         |9.11               |
|DECEPTIVE PRACTICE |799         |4.38               |
|OTHER OFFENSE      |586         |15.7               |
|ROBBERY            |508         |9.65               |
|BURGLARY           |316         |9.81               |
|WEAPONS VIOLATION  |284         |52.46              |
+-------------------+------------+-------------------+

Highest 

#### Task 4 Analysis — M1 vs M2 Comparison

**Cluster results (full 793,073 rows):**
- Overall arrest rate: **27.98%** — matches M1 Task 5 exactly (221,932 arrests / 793,072 records)
- NARCOTICS: **99.88%** arrest rate — crime type almost always results in arrest
- BURGLARY: **6.74%** — very low arrest rate; burglaries are hard to solve at scene

M2 extends M1 by adding the per-type breakdown in a single DataFrame aggregation. M1 required a dedicated job and only produced the binary `true/false` totals.

---
## Phase B: Spark MLlib — Arrest Prediction
### Tasks 5–7: Feature Engineering, Model Training, Interpretation
**Author: Wadee Feras Kharbat (ID: 230685)**

Predict whether a crime results in an arrest using four features extracted from the dataset.

### Task 5: Feature Engineering Pipeline
**Author: Wadee Feras Kharbat (ID: 230685)**

Build a Spark ML Pipeline with `StringIndexer` for categorical columns and `VectorAssembler` to combine features.

In [7]:
# ============================================
# Task 5: Feature Engineering Pipeline
# Author: Wadee Feras Kharbat (ID: 230685)
# ============================================
import time
from pyspark.sql.functions import hour, to_timestamp
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print('=== Loading Phase B Data ===')
raw_df = spark.read.csv(phase_b_input, header=True, inferSchema=True)
df_b = raw_df.withColumn('Hour', hour(to_timestamp(col('Date'), 'MM/dd/yyyy hh:mm:ss a')))
df_b = df_b.select(
    col('District'),
    col('Primary Type').alias('PrimaryType'),
    col('Hour'),
    col('Domestic').cast('string').alias('Domestic_str'),
    col('Arrest')
).dropna()
df_b = df_b.withColumn('label', col('Arrest').cast('integer'))
print('Total Rows:', df_b.count())

print('\n=== Task 5: Feature Engineering Pipeline ===')
crime_indexer    = StringIndexer(inputCol='PrimaryType',   outputCol='crime_index',    handleInvalid='skip')
domestic_indexer = StringIndexer(inputCol='Domestic_str',  outputCol='domestic_index', handleInvalid='skip')
assembler        = VectorAssembler(
    inputCols=['District', 'crime_index', 'Hour', 'domestic_index'],
    outputCol='features'
)

print('Showing sample features before training:')
temp = crime_indexer.fit(df_b).transform(df_b)
temp = domestic_indexer.fit(temp).transform(temp)
temp = assembler.transform(temp)
temp.select('PrimaryType', 'Domestic_str', 'District', 'Hour', 'features', 'label').show(5, truncate=False)

print('Feature vector layout: [District, crime_index, Hour, domestic_index]')
print('  District:       police district number (integer)')
print('  crime_index:    StringIndexer encoding of Primary Type (0 = most frequent type)')
print('  Hour:           hour of day from Date column (0-23)')
print('  domestic_index: StringIndexer encoding of Domestic flag (0 = false, 1 = true)')

train_df, test_df = df_b.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
print(f'\nTrain rows: {train_df.count()}, Test rows: {test_df.count()}')

=== Loading Phase B Data ===
Total Rows: 10000

=== Task 5: Feature Engineering Pipeline ===
Showing sample features before training:
+--------------------------+------------+--------+----+--------------------+-----+
|PrimaryType               |Domestic_str|District|Hour|features            |label|
+--------------------------+------------+--------+----+--------------------+-----+
|OFFENSE INVOLVING CHILDREN|false       |10      |3   |[10.0,12.0,3.0,0.0] |1    |
|NARCOTICS                 |false       |11      |16  |[11.0,10.0,16.0,0.0]|1    |
|ROBBERY                   |false       |14      |9   |[14.0,7.0,9.0,0.0]  |1    |
|CRIM SEXUAL ASSAULT       |false       |1       |10  |[1.0,25.0,10.0,0.0] |0    |
|CRIMINAL DAMAGE           |false       |1       |17  |[1.0,2.0,17.0,0.0]  |0    |
+--------------------------+------------+--------+----+--------------------+-----+

Feature vector layout: [District, crime_index, Hour, domestic_index]
  District:       police district number (integer

### Task 6: Train and Evaluate Three Models
**Author: Wadee Feras Kharbat (ID: 230685)**

Train Logistic Regression, Random Forest, and GBT classifiers. Evaluate AUC-ROC, Accuracy, F1, Precision, Recall, and Confusion Matrix for each.

In [8]:
# ============================================
# Task 6: Train and Evaluate Three Models
# Author: Wadee Feras Kharbat (ID: 230685)
# ============================================

binary_eval = BinaryClassificationEvaluator(labelCol='label')
mc_eval     = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction')

def evaluate_model(model_name, predictions, train_time):
    auc  = binary_eval.evaluate(predictions)
    acc  = mc_eval.evaluate(predictions, {mc_eval.metricName: 'accuracy'})
    f1   = mc_eval.evaluate(predictions, {mc_eval.metricName: 'f1'})
    prec = mc_eval.evaluate(predictions, {mc_eval.metricName: 'weightedPrecision'})
    rec  = mc_eval.evaluate(predictions, {mc_eval.metricName: 'weightedRecall'})
    print(f'\n--- {model_name} Metrics ---')
    print(f'  Training Time: {train_time:.1f}s')
    print(f'  AUC-ROC:   {auc:.4f}')
    print(f'  Accuracy:  {acc:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'\n--- Confusion Matrix ({model_name}) ---')
    predictions.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()
    return auc, acc, f1, prec, rec

print('=== Task 6: Train and Evaluate Three Models ===')

# 1. Logistic Regression
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100, regParam=0.01)
pipeline_lr = Pipeline(stages=[crime_indexer, domestic_indexer, assembler, lr])
t0 = time.time()
model_lr = pipeline_lr.fit(train_df)
t_lr = time.time() - t0
preds_lr = model_lr.transform(test_df)
metrics_lr = evaluate_model('Logistic Regression', preds_lr, t_lr)

# 2. Random Forest
rf = RandomForestClassifier(featuresCol='features', labelCol='label', numTrees=100, maxDepth=5, seed=42)
pipeline_rf = Pipeline(stages=[crime_indexer, domestic_indexer, assembler, rf])
t0 = time.time()
model_rf = pipeline_rf.fit(train_df)
t_rf = time.time() - t0
preds_rf = model_rf.transform(test_df)
metrics_rf = evaluate_model('Random Forest', preds_rf, t_rf)

# 3. GBT
gbt = GBTClassifier(featuresCol='features', labelCol='label', maxIter=50, maxDepth=5, seed=42)
pipeline_gbt = Pipeline(stages=[crime_indexer, domestic_indexer, assembler, gbt])
t0 = time.time()
model_gbt = pipeline_gbt.fit(train_df)
t_gbt = time.time() - t0
preds_gbt = model_gbt.transform(test_df)
metrics_gbt = evaluate_model('GBT', preds_gbt, t_gbt)

print('\n=== Model Comparison Table ===')
print('=' * 65)
print(f"{'Metric':<22} {'Random Forest':>14} {'Logistic Reg':>14} {'GBT':>14}")
print('=' * 65)
print(f"{'AUC-ROC':<22} {metrics_rf[0]:>14.4f} {metrics_lr[0]:>14.4f} {metrics_gbt[0]:>14.4f}")
print(f"{'Accuracy':<22} {metrics_rf[1]:>14.4f} {metrics_lr[1]:>14.4f} {metrics_gbt[1]:>14.4f}")
print(f"{'F1 Score':<22} {metrics_rf[2]:>14.4f} {metrics_lr[2]:>14.4f} {metrics_gbt[2]:>14.4f}")
print(f"{'Training Time (s)':<22} {t_rf:>14.1f} {t_lr:>14.1f} {t_gbt:>14.1f}")
print('=' * 65)

=== Task 6: Train and Evaluate Three Models ===

--- Logistic Regression Metrics ---
  Training Time: 3.5s
  AUC-ROC:   0.6654
  Accuracy:  0.8740
  F1 Score:  0.8206
  Precision: 0.8235
  Recall:    0.8740

--- Confusion Matrix (Logistic Regression) ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1674|
|    0|       1.0|    6|
|    1|       0.0|  236|
|    1|       1.0|    5|
+-----+----------+-----+

--- Random Forest Metrics ---
  Training Time: 3.4s
  AUC-ROC:   0.7819
  Accuracy:  0.8943
  F1 Score:  0.8663
  Precision: 0.8850
  Recall:    0.8943

--- Confusion Matrix (Random Forest) ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1667|
|    0|       1.0|   13|
|    1|       0.0|  190|
|    1|       1.0|   51|
+-----+----------+-----+

--- GBT Metrics ---
  Training Time: 11.2s
  AUC-ROC:   0.7899
  Accuracy:  0.8969
  F1 Score:  0.8727
  Precision: 0.8865
  Recall:    0.8969

--- C

#### Task 6 Analysis — Model Comparison

**Cluster results** (`hdfs:///data/chicago_crimes_sample.csv`, 10,000 rows):

| Metric | Random Forest | Logistic Reg | GBT |
|---|:---:|:---:|:---:|
| AUC-ROC | 0.7787 | 0.6654 | **0.7899** |
| Accuracy | 0.8943 | 0.8740 | **0.8969** |
| F1 Score | 0.8663 | 0.8206 | **0.8727** |
| Training Time (s) | 13.8 | 18.1 | 57.9 |

**Best model**: GBT wins on AUC-ROC, Accuracy, and F1, at the cost of ~4× training time vs Random Forest.

The test set is heavily imbalanced (~87.5% no-arrest). Logistic Regression posts 87.4% accuracy but catches only **5 of 241 arrests** (TP=5, FN=236) — it essentially predicts "no arrest" everywhere. Tree models recover real signal: RF lifts TP to 51 and GBT to 60 while keeping false positives low.

### Task 7: Feature Importances & Interpretation
**Author: Wadee Feras Kharbat (ID: 230685)**

Extract feature importances from the Random Forest model and interpret the results.

In [9]:
# ============================================
# Task 7: Feature Importances & Interpretation
# Author: Wadee Feras Kharbat (ID: 230685)
# ============================================

print('=== Task 7: Feature Importances & Interpretation ===')
rf_model = model_rf.stages[-1]
feature_names = ['District', 'crime_index', 'Hour', 'domestic_index']
importances   = rf_model.featureImportances.toArray()

print('--- Feature Importances (Random Forest) ---')
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    bar = '#' * int(imp * 40)
    print(f'  {name:<18} {imp:.4f}  {bar}')

print('\n--- Interpretation ---')
print('Most important feature: crime_index (~90% importance on local; ~89% on cluster)')
print('NARCOTICS crimes arrest at ~99.88%; BURGLARY at ~6.74% (full dataset, cluster)')
print('Crime type is the dominant predictor of arrest outcome.')
print('\nWhy Logistic Regression underperforms tree-based models:')
print('LR treats crime_index as a continuous ordered variable — which is semantically')
print('incorrect for a StringIndexer categorical encoding. Tree models split on index')
print('values directly, capturing the non-linear relationship between crime type and')
print('arrest probability.')

=== Task 7: Feature Importances & Interpretation ===
--- Feature Importances (Random Forest) ---
  crime_index        0.8965  ###################################
  Hour               0.0475  #
  District           0.0341  #
  domestic_index     0.0219  

--- Interpretation ---
Most important feature: crime_index (~90% importance on local; ~89% on cluster)
NARCOTICS crimes arrest at ~99.88%; BURGLARY at ~6.74% (full dataset, cluster)
Crime type is the dominant predictor of arrest outcome.

Why Logistic Regression underperforms tree-based models:
LR treats crime_index as a continuous ordered variable — which is semantically
incorrect for a StringIndexer categorical encoding. Tree models split on index
values directly, capturing the non-linear relationship between crime type and
arrest probability.


#### Task 7 Analysis

**Cluster feature importances** (from `output/task11/run.log`):

```
crime_index        0.8898  ###################################
Hour               0.0514  ##
District           0.0362  #
domestic_index     0.0225
```

`crime_index` carries ~89% of the importance mass, which directly matches the Task 4 finding: arrest probability is driven mostly by crime type. NARCOTICS arrests at nearly 100% while BURGLARY arrests at under 7%. `Hour`, `District`, and `Domestic` add minor secondary signal.

**Does this match the Task 4 arrest-rate analysis?** Yes. The StringIndexer assigns low indices to the most frequent crime types. The tree models learn that certain index ranges (NARCOTICS, PROSTITUTION) correspond to near-certain arrest, while others (THEFT, BURGLARY, CRIMINAL DAMAGE) correspond to very low arrest probability. This is exactly the pattern the arrest-rate breakdown reveals.

In [10]:
spark.stop()
print('SparkSession stopped.')

SparkSession stopped.


---
## Phase C: Deployment Modes
### Task 9: Local Execution
**Owner: Abdulaziz AlSenani (ID: 230524)**

**Evidence file**: `output/task9/task9_local_execution.txt`

```
Spark Version: 4.1.1
Master: local[*]
Input: data/chicago_crimes_sample.csv
Total Rows: 10000
```

Full Phase A (Tasks 1-4) and Phase B (Tasks 5-7) pipeline ran successfully on a MacBook Pro with `local[*]` mode using the 10,000-row sample CSV. The notebook cells above embed this local execution output directly.

### Task 10: Cluster Execution — Client Mode
**Owner: Abdulaziz AlSenani (ID: 230524)**

**Evidence file**: `output/task10/task10_cluster_client.log`

```
Master: yarn
Input: hdfs:///data/chicago_crimes.csv
Real Row Count: 793073
Application ID: application_1778738889964_0069
Spark Version: 3.5.4
```

**Top 10 Crime Types on Full HDFS Dataset**:

```
+-------------------+------+
|Primary Type       |count |
+-------------------+------+
|THEFT              |162688|
|BATTERY            |151930|
|CRIMINAL DAMAGE    |91241 |
|NARCOTICS          |74127 |
|ASSAULT            |54070 |
|MOTOR VEHICLE THEFT|48494 |
|BURGLARY           |39872 |
|OTHER OFFENSE      |36893 |
|ROBBERY            |30991 |
|DECEPTIVE PRACTICE |30396 |
+-------------------+------+
```

**Overall Arrest Rate**: 27.98% — matches M1 exactly.

### Task 11: Cluster Execution — spark-submit (YARN Cluster Mode)
**Owner: Abdulaziz AlSharif (ID: 230055)**

**Evidence file**: `output/task11/run.log`

```
Application ID:      application_1778738889964_0046
ApplicationMaster:   worker-node-1
Final status:        SUCCEEDED
Runtime:             ~3 min 37 s (13:31:56 to 13:35:33 UTC, 2026-05-21)
Master (driver):     yarn
Spark Version:       3.5.4
```

**spark-submit command**:

```bash
spark-submit \
    --master yarn \
    --deploy-mode cluster \
    --driver-memory 1024m \
    --num-executors 1 \
    --executor-memory 1g \
    --executor-cores 1 \
    --conf spark.driver.maxResultSize=128m \
    --conf spark.yarn.am.memoryOverhead=256 \
    --conf spark.yarn.appMasterEnv.PYSPARK_PYTHON=python3.12 \
    --conf spark.executorEnv.PYSPARK_PYTHON=python3.12 \
    m2_spark_ml.py
```

**Cluster ML results** (from `yarn logs -applicationId application_1778738889964_0046`):

| Metric | Random Forest | Logistic Reg | GBT |
|---|:---:|:---:|:---:|
| AUC-ROC | 0.7787 | 0.6654 | **0.7899** |
| Accuracy | 0.8943 | 0.8740 | **0.8969** |
| F1 Score | 0.8663 | 0.8206 | **0.8727** |
| Training Time (s) | 13.8 | 18.1 | 57.9 |

**Feature importances** (cluster, Random Forest):

```
crime_index        0.8898  ###################################
Hour               0.0514  ##
District           0.0362  #
domestic_index     0.0225
```

`--deploy-mode cluster` is required because the master VM is small (~4 GB) and shared with Hadoop daemons; running the driver in client mode is OOM-killed. The driver runs on `worker-node-1` in cluster mode.